In [1]:
import gymnasium as gym
import numpy as np
from tetris_gymnasium.envs.tetris import Tetris
from tetris_gymnasium.wrappers.observation import RgbObservation, FeatureVectorObservation
from tetris_gymnasium.wrappers.grouped import GroupedActionsObservations
from gymnasium.wrappers import TimeLimit, ResizeObservation, RecordVideo, FrameStackObservation, GrayscaleObservation
from stable_baselines3 import DQN, PPO
import os

In [2]:
RENDER_ENV = False
LOAD_MODEL = False
new_size = (96,136) #(84,84)
batch_size = 32
num_episodes = 4680 #43200
max_episode_steps = 100
num_stacked_frames = 4
intervals = 4
Model = "PPO" # DQN o PPO
version = 3

In [3]:
def get_last_modified_file(directory_path):
    if not os.path.isdir(directory_path):
        print(f"Error: Directory '{directory_path}' does not exist.")
        return None
    files = [os.path.join(directory_path, f) for f in os.listdir(directory_path) if os.path.isfile(os.path.join(directory_path, f))]
    if not files:
        return None
    files.sort(key=os.path.getmtime, reverse=True)
    return files[0]

target_directory = f"../Models_Saves/{Model}"  # Replace with your directory path
model_load_path = get_last_modified_file(target_directory)

if model_load_path:
    print(f"The last modified file is: {model_load_path}")
else:
    print("No files found in the directory or directory does not exist.")

The last modified file is: ../Models_Saves/PPO/PPO_V2_S2808000.zip


In [4]:
try:
    os.mkdir("../Models_Saves")
except Exception as e:
    print(f"Error: {e}")
try:
    os.mkdir("../Models_Saves/PPO")
except Exception as e:
    print(f"Error: {e}")
try:
    os.mkdir("../Models_Saves/DQN")
except Exception as e:
    print(f"Error: {e}")
try:
    os.mkdir("../Video_Tetris_IA")
except Exception as e:
    print(f"Error: {e}")
try:
    os.mkdir("../Video_Tetris_IA/PPO")
except Exception as e:
    print(f"Error: {e}")
try:
    os.mkdir("../Video_Tetris_IA/DQN")
except Exception as e:
    print(f"Error: {e}")

Error: [Errno 17] File exists: '../Models_Saves'
Error: [Errno 17] File exists: '../Models_Saves/PPO'
Error: [Errno 17] File exists: '../Models_Saves/DQN'
Error: [Errno 17] File exists: '../Video_Tetris_IA'
Error: [Errno 17] File exists: '../Video_Tetris_IA/PPO'
Error: [Errno 17] File exists: '../Video_Tetris_IA/DQN'


In [ ]:
def calc_max_height(mat, height):
  mat1 = np.rot90(mat)
  mat1 = np.rot90(mat1)
  act_height=0
  for row in mat:
    act_height+=1
    if act_height < height-4:
      continue
    count = 0
    for col in row:
      if col > 0:
        count+=1
    if count == 0:
      return act_height-1
  return height

def calc_holes(mat,height):
  mat1 = np.rot90(mat)
  holes = 0
  for row in mat1:
    for i in range(len(row)):
      if row[i] == 0:
        holes += 1
      if i == height:
        break
  return holes

def calc_adj_col(mat, height):
  mat1 = np.rot90(mat)
  mat1 = np.rot90(mat1)
  prev_height = -1
  dif_count = 0
  for row in mat1:
    act_height = 0
    for i in range(len(row)):
      if row[i] > 0:
        act_height = i
      if i==height: 
        break
    if prev_height >= 0:
      dif_count+=(abs(prev_height-act_height))
    prev_height = act_height
  return dif_count


In [6]:
try:
  env.close()
except:
  print('no hay env para cerrar')

no hay env para cerrar


In [7]:

class CustomRewardWrapper(gym.RewardWrapper):
    def __init__(self, env, holes_penalty = 0.005, height_penalty = 0.05, increase_height_penalty = 0.15, increase_holes_penalty = 0.1, dif_heigh_penalty = 0.2):
        super(CustomRewardWrapper, self).__init__(env)
        self.holes_penalty = holes_penalty
        self.height_penalty = height_penalty
        self.increase_height_penalty = increase_height_penalty
        self.increase_holes_penalty = increase_holes_penalty
        self.dif_heigh_penalty = dif_heigh_penalty

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)

        game_variables = self.env.unwrapped.get_state().board[:-4, 4:-4]
        self.previous_max_height = calc_max_height(game_variables, 1)
        self.previous_holes = calc_holes(game_variables, self.previous_max_height)

        return obs, info

    def reward(self, reward):
        #print(f"Reward original: {reward}")
        # Probar mayor penalizacion de agujeros
        custom_reward = reward
        game_variables = self.env.unwrapped.get_state().board[:-4, 4:-4]

        if game_variables.any():
            # Calcula la altura maxima actual
            current_max_height = calc_max_height(game_variables, self.previous_max_height)
            # Calcula ls espacios vacios entre la base y la altura
            current_holes = calc_holes(game_variables, current_max_height)
            #Calcula la diferencia de altura entre las columnas adyacentes y las suma
            current_height_dif = calc_adj_col(game_variables, current_max_height)
            #Penalizacion constante por altura
            custom_reward -= current_max_height*self.height_penalty
            #Penalizacion constante por agujeros
            custom_reward -= current_holes*self.holes_penalty
            #Penalizacion constante por diferencia de altura entre columnas
            custom_reward -= current_height_dif*self.dif_heigh_penalty
            #Penalizacion y recompensa por aumentar altura o disminuir altura respectivamente
            if current_max_height-self.previous_max_height != 0:
                custom_reward -= (current_max_height-self.previous_max_height)*self.increase_height_penalty
            #Penalizacion y recompensa por aumentar agujeros o disminuir agujeros respectivamente
            if current_holes-self.previous_holes != 0:
                custom_reward -= (current_holes-self.previous_holes)*self.increase_holes_penalty
            #Actualiza valores previos para el siguiente paso
            self.previous_holes = current_holes
            self.previous_max_height = current_max_height
        return custom_reward

In [8]:
def make_env(*, game, max_episode_steps=4500, **kwargs):
    env = gym.make(game, **kwargs)
    env = RgbObservation(env)
    #env = FeatureVectorObservation(env)
    env = ResizeObservation(env, new_size)
    env = GrayscaleObservation(env)
    env = FrameStackObservation(env, stack_size=num_stacked_frames)
    env = CustomRewardWrapper(env)
    env.reset(seed=42)
    return env

In [9]:


if __name__ == "__main__":
    env = make_env(game = "tetris_gymnasium/Tetris", render_mode="rgb_array")
    if Model == "DQN":
        model = DQN("CnnPolicy", env, verbose=1, buffer_size=10000) # para Mlp usar FeatureVectorObservation para Cnn usar RgbObservation
    else:
        model = PPO("CnnPolicy", env, verbose=1) # para Mlp usar FeatureVectorObservation para Cnn
    if LOAD_MODEL:
        model.load(model_load_path)
    model.learn(total_timesteps=num_episodes*max_episode_steps, log_interval=4)
    model.save(f"../Models_Saves/{Model}/{Model}_V{version}_S{max_episode_steps*num_episodes}")
    env.close()

TypeError: only integer scalar arrays can be converted to a scalar index

In [ ]:
try:
  env = RecordVideo(
    env,
    video_folder=f'../Video_Tetris_IA/{Model}',    # Folder to save videos
    name_prefix=f'{Model}_eval-Trained_steps_{num_episodes*max_episode_steps}',               # Prefix for video filenames
    episode_trigger=lambda x: True    # Record every episode
  )
except Exception as e:
  print(f'error implementando grabacion: {e}')

/home/seba/Documentos/AI_Juegos/.venv/lib/python3.10/site-packages/gymnasium/wrappers/rendering.py:293: UserWarning: WARN: Overwriting existing videos at /home/seba/Documentos/AI_Juegos/Video_Tetris_IA/PPO folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


In [ ]:
for episode in range(10):
  state, info = env.reset()
  total_reward = 0
  done = False
  step_count = 0
  while not done:
    step_count+=1
    action, _states = model.predict(state, deterministic=True)
    state, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated
    total_reward += reward
  print(f"Episode: {episode} Reward: {total_reward} Steps: {step_count}")

Episode: 0 Reward: -5.200000000000003 Steps: 37
Episode: 1 Reward: -5.520000000000006 Steps: 43
Episode: 2 Reward: -6.400000000000006 Steps: 45
Episode: 3 Reward: -5.800000000000005 Steps: 52
Episode: 4 Reward: -2.96 Steps: 45
Episode: 5 Reward: -8.600000000000007 Steps: 57
Episode: 6 Reward: -7.800000000000006 Steps: 55
Episode: 7 Reward: -10.52000000000001 Steps: 69
Episode: 8 Reward: -12.200000000000012 Steps: 67
Episode: 9 Reward: -11.20000000000001 Steps: 71


In [ ]:
# var = len(env.unwrapped.get_state().board)
# count = var
# temp = env.unwrapped.get_state().board[:-4, 4:-4]
# print(temp)
# print(calc_max_height(temp))
# print(calc_holes(temp))
# #print(len(env.unwrapped.get_state().board))

In [ ]:
# from stable_baselines3.common.env_checker import check_env
# check_env(env)